# Llama 3.1 8B Instruct - Prompting Techniques for Reasoning

Comparing **zero-shot**, **few-shot**, **few-shot CoT**, and **zero-shot CoT** prompting on the `facebook/natural_reasoning` dataset.

**Model**: `meta-llama/Llama-3.1-8B-Instruct` (4-bit quantized)  

In [ ]:
!pip install transformers accelerate bitsandbytes
!pip install rouge-score tqdm

In [ ]:
import json
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
from huggingface_hub import login
from google.colab import drive, userdata, files

In [ ]:
login(token=userdata.get("HF_TOKEN"))
print("Logged into HuggingFace")

## Configuration

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DRIVE_DATA_PATH = "/content/drive/MyDrive/NLP_Project/Data"
MAX_NEW_TOKENS = 1024
SEED = 42

TECHNIQUES = ["zero_shot", "few_shot", "few_shot_cot", "zero_shot_cot"]

## Dataset

In [ ]:
drive.mount('/content/drive')

with open(f'{DRIVE_DATA_PATH}/sampled.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

print(f"Loaded {len(dataset)} questions")

In [ ]:
df_data = pd.DataFrame(dataset)
print(df_data['answer_type'].value_counts())
print()
df_data[['sample_id', 'answer_type', 'question']].head(10)

## Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Prompting Techniques

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "A ball is dropped from a height of 20 meters. Ignoring air resistance, how long does it take to reach the ground? Use g = 10 m/s².",
        "answer": "2 seconds",
        "reasoning": "Using the kinematic equation h = ½gt², where h = 20 m and g = 10 m/s². Substituting: 20 = ½ × 10 × t², which simplifies to 20 = 5t². Solving: t² = 4, so t = 2 seconds."
    },
    {
        "question": "Find the derivative of f(x) = 3x² + 2x - 5.",
        "answer": "f'(x) = 6x + 2",
        "reasoning": "Applying the power rule term by term: the derivative of 3x² is 2·3x = 6x, the derivative of 2x is 2, and the derivative of the constant -5 is 0. Combining: f'(x) = 6x + 2."
    },
    {
        "question": "If a number is divisible by 6, must it also be divisible by 3? Explain why.",
        "answer": "Yes, because 6 = 2 × 3, any multiple of 6 is automatically a multiple of 3.",
        "reasoning": "Since 6 = 2 × 3, any number divisible by 6 can be written as 6k = 2 × 3 × k for some integer k. This equals 3 × (2k), which is clearly a multiple of 3. Therefore divisibility by 6 guarantees divisibility by 3."
    }
]

In [ ]:
def build_messages(question, technique):
    if technique == "zero_shot":
        return [
            {"role": "system", "content": "You are a helpful assistant. Answer the question directly and concisely."},
            {"role": "user", "content": question}
        ]

    elif technique == "few_shot":
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Answer questions directly and concisely."}
        ]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": ex["answer"]})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "few_shot_cot":
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Think step by step before giving your final answer."}
        ]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"{ex['reasoning']}\n\nTherefore, the answer is: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "zero_shot_cot":
        return [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{question}\n\nLet's think step by step."}
        ]

## Inference

In [ ]:
def generate_response(messages):
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    if isinstance(inputs, torch.Tensor):
        input_ids = inputs.to(model.device)
    else:
        input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:], skip_special_tokens=True
    )
    return response.strip()

In [ ]:
results = []

for technique in TECHNIQUES:
    print(f"\n{'='*60}")
    print(f"Running: {technique}")
    print(f"{'='*60}")

    for item in tqdm(dataset, desc=technique):
        messages = build_messages(item["question"], technique)

        start = time.time()
        response = generate_response(messages)
        gen_time = time.time() - start

        results.append({
            "sample_id": item["sample_id"],
            "question": item["question"],
            "reference_answer": item["reference_answer"],
            "answer_type": item["answer_type"],
            "technique": technique,
            "response": response,
            "generation_time": gen_time,
            "response_length": len(response.split()),
        })

df = pd.DataFrame(results)
print(f"\nDone: {len(df)} results ({len(dataset)} questions x {len(TECHNIQUES)} techniques)")

In [ ]:
# Optional: download results as JSON
df.to_json("/content/results_llama.json", orient="records", indent=2)
files.download("/content/results_llama.json")

## Evaluation

In [ ]:
def normalize(text):
    return text.lower().strip().rstrip('.')

def exact_match(prediction, reference):
    return normalize(prediction) == normalize(reference)

def contains_match(prediction, reference):
    return normalize(reference) in normalize(prediction)

def compute_rouge_l(prediction, reference):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    return scorer.score(reference, prediction)['rougeL'].fmeasure

In [ ]:
has_ref = df[df['answer_type'] != 'no_answer'].copy()

has_ref['exact_match'] = has_ref.apply(
    lambda r: exact_match(r['response'], r['reference_answer']), axis=1
)
has_ref['contains_match'] = has_ref.apply(
    lambda r: contains_match(r['response'], r['reference_answer']), axis=1
)
has_ref['rouge_l'] = has_ref.apply(
    lambda r: compute_rouge_l(r['response'], r['reference_answer']), axis=1
)

metrics = has_ref.groupby('technique').agg(
    exact_match=('exact_match', 'mean'),
    contains_match=('contains_match', 'mean'),
    rouge_l=('rouge_l', 'mean'),
    avg_response_len=('response_length', 'mean'),
    avg_gen_time=('generation_time', 'mean'),
).round(4)

metrics

## Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics['exact_match'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Exact Match')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)

metrics['contains_match'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Contains Match')
axes[1].set_ylim(0, 1)

metrics['rouge_l'].plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('ROUGE-L')
axes[2].set_ylim(0, 1)

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Llama 3.1 8B — Performance by Prompting Technique', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='response_length', by='technique', ax=axes[0])
axes[0].set_title('Response Length (words)')
axes[0].set_xlabel('')
plt.sca(axes[0])
plt.xticks(rotation=45, ha='right')

df.groupby('technique')['generation_time'].mean().plot(
    kind='bar', ax=axes[1], color='mediumpurple'
)
axes[1].set_title('Avg Generation Time (seconds)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Performance breakdown by answer type
breakdown = has_ref.groupby(['technique', 'answer_type']).agg(
    contains_match=('contains_match', 'mean'),
    count=('contains_match', 'count'),
).round(4)

pivot = breakdown.reset_index().pivot(
    index='answer_type', columns='technique', values='contains_match'
)

pivot.plot(kind='bar', figsize=(10, 5))
plt.title('Contains Match by Answer Type')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.legend(title='Technique', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

### Qualitative Comparison

In [ ]:
def show_comparison(sample_id):
    subset = df[df['sample_id'] == sample_id]
    row0 = subset.iloc[0]
    print(f"Question: {row0['question'][:300]}")
    ref = row0['reference_answer']
    print(f"Reference: {ref[:200] if ref else '(none)'}")
    print('-' * 80)
    for _, row in subset.iterrows():
        print(f"\n[{row['technique']}] ({row['response_length']} words, {row['generation_time']:.1f}s)")
        print(row['response'][:500])
    print('\n' + '=' * 80 + '\n')

sample_ids = df[df['answer_type'].isin(['short', 'single_word'])]['sample_id'].unique()[:3]
for sid in sample_ids:
    show_comparison(sid)

### CoT Impact Analysis

Pairwise comparison: how often does adding chain-of-thought fix or break answers?

In [ ]:
comparisons = [
    ("zero_shot", "zero_shot_cot", "Zero-Shot vs Zero-Shot CoT"),
    ("few_shot", "few_shot_cot", "Few-Shot vs Few-Shot CoT"),
    ("zero_shot_cot", "few_shot_cot", "Zero-Shot CoT vs Few-Shot CoT"),
]

for t1, t2, title in comparisons:
    df1 = has_ref[has_ref['technique'] == t1].set_index('sample_id')['contains_match']
    df2 = has_ref[has_ref['technique'] == t2].set_index('sample_id')['contains_match']

    both_right = (df1 & df2).sum()
    only_t1 = (df1 & ~df2).sum()
    only_t2 = (~df1 & df2).sum()
    both_wrong = (~df1 & ~df2).sum()

    print(f"\n{title}")
    print(f"  Both correct:  {both_right}")
    print(f"  Only {t1}: {only_t1}")
    print(f"  Only {t2}: {only_t2}")
    print(f"  Both wrong:    {both_wrong}")

In [ ]:
pivot_correct = has_ref.pivot(
    index='sample_id', columns='technique', values='contains_match'
).astype(int)

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(pivot_correct, cmap='RdYlGn', cbar_kws={'label': 'Correct'},
            xticklabels=True, yticklabels=False, ax=ax)
ax.set_title('Correctness by Question and Technique')
ax.set_xlabel('Technique')
ax.set_ylabel('Questions')
plt.tight_layout()
plt.show()